# 03 — Azure AutoML Training

## Objective
Train a time-series forecasting model using Azure AutoML to predict
the next 90 days of store sales by product family.

**What Azure AutoML does for us:**
- Automatically tries 10+ model architectures (ARIMA, Prophet, LightGBM, etc.)
- Handles feature engineering (lags, rolling windows, date features)
- Performs hyperparameter tuning across all model types
- Ranks models on a leaderboard by our chosen metric
- Provides model explainability (feature importance)

**Why AutoML vs manual model building?**
- In production, AutoML finds competitive models faster than manual tuning
- The leaderboard comparison is an artifact you can show stakeholders
- Explainability is built-in, not bolted on
- It demonstrates the ML lifecycle, not just model.fit()

In [ ]:
# === Setup ===
import os
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input, automl
from azure.ai.ml.entities import AmlCompute, Data
from azure.ai.ml.constants import AssetTypes

import pandas as pd
import json

print('Azure ML SDK v2 loaded successfully.')
print(f'Subscription: {os.getenv("AZURE_SUBSCRIPTION_ID", "NOT SET")}')
print(f'Workspace: {os.getenv("AML_WORKSPACE_NAME", "NOT SET")}')

## 1. Connect to Azure ML Workspace

We use `DefaultAzureCredential` which tries multiple auth methods:
1. Environment variables → 2. Managed identity → 3. Azure CLI → 4. Browser login

For local development, make sure you've run `az login` first.

In [ ]:
# Authenticate and connect to the workspace
try:
    # Try DefaultAzureCredential first (works if az login was run)
    credential = DefaultAzureCredential()
    ml_client = MLClient(
        credential=credential,
        subscription_id=os.getenv('AZURE_SUBSCRIPTION_ID'),
        resource_group_name=os.getenv('AZURE_RESOURCE_GROUP'),
        workspace_name=os.getenv('AML_WORKSPACE_NAME'),
    )
    
    # Test the connection
    ws = ml_client.workspaces.get(os.getenv('AML_WORKSPACE_NAME'))
    print(f'✓ Connected to: {ws.name}')
    print(f'  Region: {ws.location}')
    print(f'  Resource Group: {os.getenv("AZURE_RESOURCE_GROUP")}')
    
except Exception as e:
    print(f'DefaultAzureCredential failed: {e}')
    print('Falling back to interactive browser login...')
    credential = InteractiveBrowserCredential()
    ml_client = MLClient(
        credential=credential,
        subscription_id=os.getenv('AZURE_SUBSCRIPTION_ID'),
        resource_group_name=os.getenv('AZURE_RESOURCE_GROUP'),
        workspace_name=os.getenv('AML_WORKSPACE_NAME'),
    )

## 2. Create Compute Cluster

AutoML needs a compute target to run training on. We create an auto-scaling cluster:
- **Scales to 0** when idle (no cost when not training)
- **Standard_DS3_v2**: ~$0.24/hr per node — good price/performance
- **4 max nodes**: Allows parallel model evaluation

In [ ]:
# Compute configuration from environment
COMPUTE_NAME = os.getenv('AML_COMPUTE_NAME', 'forecast-cluster')
COMPUTE_SIZE = os.getenv('AML_COMPUTE_SIZE', 'Standard_DS3_v2')
MIN_NODES = int(os.getenv('AML_COMPUTE_MIN_NODES', '0'))
MAX_NODES = int(os.getenv('AML_COMPUTE_MAX_NODES', '4'))

try:
    # Check if compute exists
    compute = ml_client.compute.get(COMPUTE_NAME)
    print(f'✓ Compute exists: {COMPUTE_NAME} ({compute.size})')
except:
    # Create the cluster
    print(f'Creating compute cluster: {COMPUTE_NAME}...')
    compute = AmlCompute(
        name=COMPUTE_NAME,
        size=COMPUTE_SIZE,
        min_instances=MIN_NODES,
        max_instances=MAX_NODES,
        idle_time_before_scale_down=300,  # 5 min idle -> scale down
    )
    ml_client.compute.begin_create_or_update(compute).result()
    print(f'✓ Created: {COMPUTE_NAME} ({COMPUTE_SIZE}, {MIN_NODES}-{MAX_NODES} nodes)')

## 3. Upload Training Data

We register our prepared data as a versioned Azure ML dataset.
This provides data lineage — you can trace which data trained which model.

In [ ]:
from pathlib import Path

data_path = Path('../data/processed/automl_training_data.parquet')

# Register as versioned data asset
data_asset = Data(
    name='store-sales-training',
    description='Favorita store sales with engineered features for forecasting',
    path=str(data_path),
    type=AssetTypes.URI_FILE,
)

registered_data = ml_client.data.create_or_update(data_asset)
print(f'✓ Data registered: {registered_data.name} v{registered_data.version}')
print(f'  Size: {data_path.stat().st_size / (1024**2):.1f} MB')

# Create input reference for AutoML
training_data_input = Input(
    type=AssetTypes.URI_FILE,
    path=f'azureml:{registered_data.name}:{registered_data.version}'
)

## 4. Configure AutoML Forecasting

### Key Configuration Decisions

| Setting | Value | Rationale |
|---------|-------|-----------|
| `target_column` | `sales` | Daily unit sales is what we forecast |
| `primary_metric` | `normalized_root_mean_squared_error` | Scale-independent; works across stores |
| `forecast_horizon` | 90 days | Covers 30/60/90 day business planning windows |
| `time_series_id` | `[store_nbr, family]` | Each store×product is a unique series |
| `target_lags` | auto | Let AutoML find optimal lag features |
| `enable_dnn_training` | True | Include deep learning (TCN) models |
| `max_trials` | 50 | Enough models for a thorough search |

In [ ]:
# Configure the forecasting job
FORECAST_HORIZON = int(os.getenv('FORECAST_HORIZON', '90'))
PRIMARY_METRIC = os.getenv('PRIMARY_METRIC', 'normalized_root_mean_squared_error')
TIMEOUT_HOURS = int(os.getenv('TRAINING_TIMEOUT_HOURS', '2'))

forecasting_job = automl.forecasting(
    training_data=training_data_input,
    target_column_name='sales',
    primary_metric=PRIMARY_METRIC,
    compute=COMPUTE_NAME,
    experiment_name=os.getenv('EXPERIMENT_NAME', 'store-sales-forecasting'),
    training=automl.ForecastingTrainingConfig(
        enable_dnn_training=True,
        enable_model_explainability=True,
        allowed_training_algorithms=[
            'LightGBM', 'XGBRegressor', 'Prophet',
            'AutoArima', 'ExponentialSmoothing',
            'TCNForecaster', 'ElasticNet',
            'DecisionTree', 'RandomForest',
        ],
    ),
)

# Forecasting-specific settings
forecasting_job.set_forecast_settings(
    time_column_name='date',
    time_series_id_column_names=['store_nbr', 'family'],
    forecast_horizon=FORECAST_HORIZON,
    frequency='D',
    target_lags='auto',
    target_rolling_window_size='auto',
)

# Resource limits (prevent runaway costs)
forecasting_job.set_limits(
    timeout_minutes=TIMEOUT_HOURS * 60,
    trial_timeout_minutes=30,
    max_trials=50,
    max_concurrent_trials=MAX_NODES,
    enable_early_termination=True,
)

# Featurization
forecasting_job.set_featurization(mode='auto')

print('✓ AutoML forecasting configured:')
print(f'  Target: sales | Metric: {PRIMARY_METRIC}')
print(f'  Horizon: {FORECAST_HORIZON} days | Series: store_nbr × family')
print(f'  Max trials: 50 | Timeout: {TIMEOUT_HOURS}hr')

## 5. Submit the Experiment

When submitted, Azure AutoML will:
1. Provision compute nodes (may take 2-5 min if scaling from 0)
2. Profile the data (distributions, correlations)
3. Run featurization (encoding, imputation, lag generation)
4. Train models in parallel across cluster nodes
5. Rank all models and select the winner

**Monitor in Azure ML Studio** — the URL is printed below.

In [ ]:
# Submit the training job
submitted_job = ml_client.jobs.create_or_update(forecasting_job)

print(f'✓ Job submitted: {submitted_job.name}')
print(f'  Status: {submitted_job.status}')
print(f'\n  📊 Monitor in Azure ML Studio:')
print(f'  {submitted_job.studio_url}')
print(f'\n  ⏳ Training typically takes 1-2 hours.')
print(f'  You can close this notebook — the experiment runs in Azure.')

In [ ]:
# Wait for completion (optional — you can also check Azure ML Studio)
import time

print('Waiting for experiment to complete...')
while True:
    job = ml_client.jobs.get(submitted_job.name)
    status = job.status
    if status in ['Completed', 'Failed', 'Canceled']:
        break
    print(f'  Status: {status}', end='\r')
    time.sleep(60)

print(f'\n✓ Experiment {status}!')
if status == 'Completed':
    print(f'\nNext step: 04_model_evaluation.ipynb')
    print(f'Job name to use: {submitted_job.name}')

## 6. Save Job Name

Save the job name for use in subsequent notebooks and scripts.

In [ ]:
# Save job name for downstream notebooks
job_info = {
    'job_name': submitted_job.name,
    'studio_url': submitted_job.studio_url,
    'status': submitted_job.status,
}

with open('../data/output/automl_job_info.json', 'w') as f:
    json.dump(job_info, f, indent=2)

print(f'Job info saved to data/output/automl_job_info.json')
print(f'Use this in notebooks 04 and 05: --job-name {submitted_job.name}')